In [0]:
from pyspark.sql import functions as F
import logging
import sys
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, FloatType
import uuid
from datetime import datetime, timezone
from pyspark.sql.window import Window
from delta.tables import DeltaTable

logger = logging.getLogger("turbines")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
logger.propagate = False

run_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)

logger.info(f"Silver started at {started_at} with pipeline id {run_id}")

In [0]:
def log_run(layer, started_at, rows_in, rows_out, status="SUCCESS", message=None):
    spark.createDataFrame(
        [(run_id, layer, started_at, datetime.now(timezone.utc), rows_in, rows_out, status, message)],
        "run_id string, layer string, started_at timestamp, finished_at timestamp, rows_in long, rows_out long, status string, message string"
    ).write.mode("append").saveAsTable("turbines.control.pipeline_log")

In [0]:
max_date = "2022-03-30"    # remove for next run
SOURCE = "turbine_readings"

In [0]:
wm = (spark.table("turbines.control.watermarks")
        .filter(F.col("source") == SOURCE)
        .agg(F.max("watermark_ts")).first()[0])
logger.info(f"Watermark: {wm}")   

In [0]:
df = spark.table("turbines.bronze.raw")
if wm is not None:
    df = df.filter(F.col("timestamp") > F.lit(wm))
if max_date is not None:
    df = df.filter(F.to_date("timestamp") <= max_date)
n_in = df.count()

In [0]:
w = Window.partitionBy("turbine_id", "timestamp").orderBy(F.col("ingest_ts").desc())
df = (df.withColumn("rn", F.row_number().over(w))
        .filter("rn = 1").drop("rn"))

In [0]:
bad_cond = (F.col("power_output") < 0) | (F.col("power_output") > 4.5)   
df_bad  = df.filter(bad_cond)
df_good = df.filter(~bad_cond | F.col("power_output").isNull())   

(df_bad.withColumn("reason", F.lit("power_output outside physical bounds"))
       .withColumn("run_id", F.lit(run_id))
       .write.mode("append").saveAsTable("turbines.silver.quarantine"))

In [0]:
means = df_good.groupBy("turbine_id").agg(F.avg("power_output").alias("mean_po"))
df_clean = (df_good.join(means, "turbine_id", "left")
    .withColumn("power_output", F.coalesce("power_output", "mean_po"))
    .drop("mean_po"))

In [0]:
if not spark.catalog.tableExists("turbines.silver.readings"):
    df_clean.write.saveAsTable("turbines.silver.readings")
    logger.info("created silver.readings (first run)")
else:
    (DeltaTable.forName(spark, "turbines.silver.readings").alias("t")
        .merge(df_clean.alias("s"),
               "t.turbine_id = s.turbine_id AND t.timestamp = s.timestamp")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
new_wm = df_clean.agg(F.max("timestamp")).first()[0]
spark.createDataFrame([(SOURCE, new_wm, datetime.now(timezone.utc))],
    "source string, watermark_ts timestamp, updated_at timestamp") \
    .write.mode("append").saveAsTable("turbines.control.watermarks")

In [0]:
n_silver = spark.table("turbines.silver.readings").count()
logger.info(f"silver.readings: {n_silver} rows")
spark.table("turbines.silver.readings").show(5)
log_run("silver", started_at, n_in, n_silver)